# Iris Flower Classification with Optimized KNN 🌸

**Project type:** Supervised Machine Learning · Classification  
**Goal:** classify an Iris flower as *setosa*, *versicolor*, or *virginica* from four measurements.

## What makes this version better?
- A reproducible, stratified train/test split.
- A scikit-learn `Pipeline` so preprocessing is fitted inside each cross-validation fold.
- Feature-scaling and KNN hyperparameter tuning with `GridSearchCV`.
- A baseline-versus-tuned comparison.
- Held-out test evaluation with accuracy, precision, recall, F1-score, and a confusion matrix.
- A small prediction example for a new flower.

> **Important:** A higher score is not guaranteed. The notebook reports the measured result from the selected split; it does not force 100% accuracy.


## 1. Import libraries

If needed, install dependencies first using `pip install -r requirements.txt`.


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")


## 2. Load and validate the dataset

In [ ]:
# Keep iris.csv in the same folder as this notebook.
df = pd.read_csv("iris.csv")

expected_columns = [
    "sepal_length", "sepal_width", "petal_length", "petal_width", "species"
]
missing_columns = [column for column in expected_columns if column not in df.columns]
if missing_columns:
    raise ValueError(
        f"iris.csv is missing columns: {missing_columns}. "
        f"Found columns: {df.columns.tolist()}"
    )

df = df[expected_columns].copy()

if df.isna().any().any():
    raise ValueError("The dataset contains missing values. Please clean them before training.")

feature_columns = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
X = df[feature_columns]
y = df["species"]

print(f"Rows: {len(df)}")
print(f"Features: {len(feature_columns)}")
print(f"Classes: {sorted(y.unique())}")
display(df.head())
display(df["species"].value_counts().rename("count").to_frame())


## 3. Explore the data

In [ ]:
display(df[feature_columns].describe().round(2))

sns.pairplot(df, hue="species", diag_kind="hist")
plt.show()


## 4. Split the data

We keep 20% of the rows as a final test set. The model-selection process uses only the training set. Stratification preserves the class proportions in both sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print("\nTraining class proportions:")
display(y_train.value_counts(normalize=True).sort_index().round(3))


## 5. Establish a baseline KNN model

This baseline uses feature standardization and `k=3`. Its score is measured on the same held-out test set that will be used for the tuned model.


In [ ]:
baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=3))
])

baseline_model.fit(X_train, y_train)
baseline_predictions = baseline_model.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)

print(f"Baseline KNN test accuracy: {baseline_accuracy:.2%}")


## 6. Tune KNN with cross-validation

KNN is distance-based, so scaling matters. This search compares standard and min-max scaling, several neighbor counts, voting weights, and distance metrics. The scaler is inside the pipeline, preventing data leakage during cross-validation.

The best settings are selected using 5-fold stratified cross-validation on the training set only.


In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

param_grid = {
    "scaler": [StandardScaler(), MinMaxScaler()],
    "knn__n_neighbors": list(range(1, 26, 2)),
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "manhattan"]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
    refit=True
)

grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

print("Best parameters:")
for name, value in grid_search.best_params_.items():
    print(f"  {name}: {value}")
print(f"Best mean cross-validation accuracy: {grid_search.best_score_:.2%}")


## 7. Evaluate the tuned model on the held-out test set

In [ ]:
tuned_predictions = best_model.predict(X_test)
tuned_accuracy = accuracy_score(y_test, tuned_predictions)

results = pd.DataFrame({
    "Model": ["Baseline KNN (k=3)", "Tuned KNN"],
    "Test accuracy": [baseline_accuracy, tuned_accuracy]
})
display(results.assign(**{"Test accuracy": results["Test accuracy"].map(lambda score: f"{score:.2%}")}))

print(f"Tuned KNN test accuracy: {tuned_accuracy:.2%}")
print("\nClassification report:")
print(classification_report(y_test, tuned_predictions, zero_division=0))


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    tuned_predictions,
    labels=best_model.classes_,
    display_labels=best_model.classes_,
    values_format="d",
    cmap="Blues"
)
plt.title("Tuned KNN — Confusion Matrix")
plt.tight_layout()
plt.show()


## 8. Review the cross-validation results

These are training-set cross-validation results, not test-set results. A small gap between training and cross-validation scores is generally preferable to a model that fits training data extremely well but generalizes poorly.


In [ ]:
cv_results = pd.DataFrame(grid_search.cv_results_)
comparison = cv_results[
    [
        "param_scaler", "param_knn__n_neighbors", "param_knn__weights",
        "param_knn__metric", "mean_train_score", "mean_test_score", "std_test_score"
    ]
].copy()

comparison.columns = [
    "scaler", "k", "weights", "distance_metric",
    "mean_train_accuracy", "mean_cv_accuracy", "cv_accuracy_std"
]
comparison["scaler"] = comparison["scaler"].map(
    lambda obj: obj.__class__.__name__ if hasattr(obj, "__class__") else str(obj)
)
comparison = comparison.sort_values(
    ["mean_cv_accuracy", "cv_accuracy_std"],
    ascending=[False, True]
).reset_index(drop=True)

display(comparison.head(10).style.format({
    "mean_train_accuracy": "{:.2%}",
    "mean_cv_accuracy": "{:.2%}",
    "cv_accuracy_std": "{:.2%}"
}))


## 9. Predict a new flower

Enter measurements in centimetres. The example below is illustrative; change the values to test another flower.


In [ ]:
new_flower = pd.DataFrame([{
    "sepal_length": 5.1,
    "sepal_width": 3.5,
    "petal_length": 1.4,
    "petal_width": 0.2
}])

predicted_species = best_model.predict(new_flower[feature_columns])[0]
predicted_probabilities = best_model.predict_proba(new_flower[feature_columns])[0]
confidence = predicted_probabilities.max()

print("Predicted species:", predicted_species)
print(f"Highest class probability: {confidence:.2%}")
print("\nClass probabilities:")
display(pd.DataFrame({
    "species": best_model.classes_,
    "probability": predicted_probabilities
}).sort_values("probability", ascending=False).reset_index(drop=True))


## 10. Conclusion

- The baseline and tuned models are evaluated on the same held-out test set.
- Hyperparameters are selected with stratified cross-validation on the training set.
- Scaling is performed within the pipeline to avoid leakage between folds.
- The measured test accuracy depends on the split and dataset; 100% accuracy is not guaranteed.
- For a portfolio, report the actual scores generated when you run the notebook rather than promising a specific result.

## Run the project
1. Keep `iris.csv` in the same folder as this notebook.
2. Install dependencies from `requirements.txt`.
3. Open the notebook in Jupyter Notebook, JupyterLab, or Google Colab.
4. Run all cells from top to bottom.
